# Marginal Impact — 200 RP OEP, band-window method

**Inputs — three servers, no two of which can be linked in SQL**

| object | server | grain |
|---|---|---|
| Account-level ELT | **RMS main** | one row per (account, event) |
| **Group** portfolio-level ELT | **QBE main** | one row per event |
| YEQT + `VendorEventsMap` | **QBE sandbox** | one `LossQuantile` per (SimID, EventID) |

The marginal is measured against the **group portfolio** YLT, simulated from the
group ELT on the QBE main server — not from an RMS portfolio analysis.

**Method** (manager's 8 steps, OEP basis)

1. Simulate the group portfolio ELT against YEQT → group YELT (event level)
2. Year OEP = max event loss per SimID → sort desc → 200 RP at rank `N/200`
3. Window = the **band** of ranks `[BAND_LO, BAND_HI]`, carrying *all* their events
4. Simulate the account ELT restricted to band (year, event)
5. Subtract on (SimID, EventID)
6. Recompute per-year max — demotion-safe
7. New 200 RP read at `BAND_POS` within the re-sorted band
8. Marginal = original − new

**Two joins now cross a server boundary**, both in Python:

```
account ELT (RMS main) --VENDOREVENTID--> mapping (QBE sandbox)
group ELT   (QBE main) --  ??? key  ----> mapping (QBE sandbox)
```

The group ELT's event key is the open question — see section 2. Both sides must
land in the **same** event space before the subtraction in step 5 means anything.

## 1 · Setup

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# --- run configuration -----------------------------------------------------
N_SIMS        = None        # DERIVED from YEQT -- see section 2
RETURN_PERIOD = 200

# --- window: a BAND of ranks around the target, per the agreed methodology --
BAND_LO = 1_000            # first rank in the window (1-based, inclusive)
BAND_HI = 4_000            # last  rank in the window (1-based, inclusive)
#   width 3,001, straddling the 2,500th row.
#   Years ranked 1..BAND_LO-1 are assumed to stay above the target rank after
#   removal; years ranked > BAND_HI are assumed not to rise into it.

# --- three servers, none linkable to another -------------------------------
#   RMS main    -> account-level ELT         (rdm_account / rdm_accountstd)
#   QBE main    -> GROUP portfolio-level ELT (what the marginal is measured against)
#   QBE sandbox -> YEQT + VendorEventsMap    (gp_reference_dev)
ANLSID_ACCOUNT = 112        # RMS analysis holding the account ELTs
PERSPCODE      = "RL"       # SAME perspective on both ELTs, or nothing reconciles

GROUP_TABLE    = None       # set me: qualified table/view of the group ELT on QBE main
GROUP_FILTER   = ""         # set me: extra WHERE clause identifying the group portfolio
GROUP_EVENT_KEY = None      # set me: "VENDOREVENTID" or "EVENTID" -- see section 2

VENDORVERID   = 33          # must match the RMS model version behind ANLSID_ACCOUNT
HAZARDZONEID  = 57
YEQTID        = 69

PARAM_FLOOR = 1e-7
MU_CAP      = 1.0 - 1e-7
TOL         = 1e-6

BAND_WIDTH = BAND_HI - BAND_LO + 1

def band_geometry(n_sims: int, rp: int = RETURN_PERIOD,
                  lo: int = BAND_LO, hi: int = BAND_HI) -> tuple[int, int]:
    # (target_rank, position within the band). Nothing is hard-coded.
    rank = int(round(n_sims / rp))
    if not (lo <= rank <= hi):
        raise ValueError(f"target rank {rank:,} (from {n_sims:,} sims) falls outside "
                         f"band [{lo:,}, {hi:,}] -- move the band or check the sim count")
    return rank, rank - (lo - 1)

print(f"band = [{BAND_LO:,}, {BAND_HI:,}] (width {BAND_WIDTH:,}); "
      "target rank derived once the sim count is loaded")

band = [1,000, 4,000] (width 3,001); target rank derived once the sim count is loaded


## 2 · Extracts — three servers

**The open question: what event key does the group ELT carry?**

The account ELT comes from RMS, so its `EVENTID` is a *vendor* event id and joins
to `VendorEventsMap.VENDOREVENTID`. The group ELT comes from QBE main, which may
already hold QBE-native `EVENTID`s — in which case it joins straight to
`YEQT.EVENTID` and skips the vendor map entirely.

`inspect_group_key()` decides this from the data rather than by assumption: it
tests the group ELT's event ids against both key spaces and reports the match rate
on each. Set `GROUP_EVENT_KEY` from what it tells you. Guessing wrong does not
error — it produces a near-empty join, or worse, a partial one.

**Do not merge the full ELT against the full mapping.** Filter to one account
first. The earlier attempt merged every account at once and materialised 139.8M
rows; one account against the mapping is `account_events × sims_per_event`.

In [2]:
# from sqlalchemy import create_engine
#
# RMS_CONN = ("mssql+pyodbc://@<rms_main_server>/<rdm_database>"
#             "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
# QBE_CONN = ("mssql+pyodbc://@<qbe_main_server>/<group_database>"
#             "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
# REF_CONN = ("mssql+pyodbc://@<qbe_sandbox_server>/gp_reference_dev"
#             "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
#
# engine_rdm = create_engine(RMS_CONN)   # RMS main    -- account ELT
# engine_qbe = create_engine(QBE_CONN)   # QBE main    -- GROUP portfolio ELT
# engine_ref = create_engine(REF_CONN)   # QBE sandbox -- YEQT + VendorEventsMap

SQL_ACCOUNT_ELT = """
SELECT
      ra.ANLSID, ra.ID, ra.EVENTID AS VENDOREVENTID, ra.PERSPCODE
    , ra.PERSPVALUE                      -- mean loss
    , ras.STDDEVC, ras.STDDEVI, ras.EXPVALUE
    , ae.RATE                            -- AAL control ONLY; never applied
FROM DBO.RDM_ACCOUNT ra
INNER JOIN DBO.RDM_ACCOUNTSTD ras
       ON  ra.ANLSID      = ras.ANLSID
       AND ra.ID          = ras.ID
       AND ra.PERSPCODE   = ras.PERSPCODE
       AND ra.EVENTID     = ras.EVENTID
       AND ra.PARTITIONID = ras.PARTITIONID    -- guards partition fan-out
INNER JOIN DBO.RDM_ANLSEVENT ae
       ON ae.EVENTID = ra.EVENTID AND ae.ANLSID = ra.ANLSID
WHERE ra.ANLSID = {anlsid} AND ra.PERSPCODE = '{persp}' AND ras.EXPVALUE > 0
  {id_clause}
"""

SQL_MAP = """
SELECT yeqt.SIMID, vem.VENDOREVENTID, yeqt.EVENTID, yeqt.LossQuantile
FROM EVT.VENDOREVENTSMAP vem
INNER JOIN SIM.YEQT yeqt ON yeqt.EVENTID = vem.EVENTID
WHERE vem.HAZARDZONEID = {hz} AND vem.VENDORVERID = {vv} AND yeqt.YEQTID = {yq}
"""

DTYPES = {"SIMID": "int32", "VENDOREVENTID": "int64", "EVENTID": "int64",
          "LossQuantile": "float32"}

ELT_COLS = ["PERSPVALUE", "STDDEVC", "STDDEVI", "EXPVALUE"]


def load_n_sims(yeqtid=YEQTID):
    # Simulation-year count, derived from YEQT alone.
    #
    # Use the SIMID RANGE, not the distinct count. YEQT holds (year, event)
    # occurrences, so a year with no events has no rows and is invisible to
    # COUNT(DISTINCT SIMID). SIMIDs are a contiguous block, so max-min+1
    # recovers the true count including loss-free years.
    r = pd.read_sql(f"""
        SELECT COUNT(DISTINCT SIMID) AS n_with_events
             , MIN(SIMID) AS lo, MAX(SIMID) AS hi
        FROM SIM.YEQT WHERE YEQTID = {yeqtid}
    """, engine_ref).iloc[0]

    lo, hi = int(r["lo"]), int(r["hi"])
    n_with_events = int(r["n_with_events"])
    n_sims = hi - lo + 1

    print(f"YEQTID {yeqtid}: SIMID {lo:,} to {hi:,}")
    print(f"  range       : {n_sims:,} simulated years")
    print(f"  with events : {n_with_events:,} ({n_with_events / n_sims:.2%})")
    print(f"  loss-free   : {n_sims - n_with_events:,}")

    if n_with_events > n_sims:
        raise ValueError("distinct SIMIDs exceed the range -- SIMIDs are not contiguous")
    if n_with_events < n_sims * 0.5:
        print("  WARNING: over half the years carry no events. Confirm the SIMID "
              "block is contiguous and is a single simulation set.")
    return n_sims, lo, hi


def load_mapping(hz=HAZARDZONEID, vv=VENDORVERID, yq=YEQTID):
    """QBE sandbox. Pulled once and reused by every account and by the group."""
    df = pd.read_sql(SQL_MAP.format(hz=hz, vv=vv, yq=yq), engine_ref)
    for c, t in DTYPES.items():
        if c in df:
            df[c] = df[c].astype(t)

    dup = df.duplicated(subset=["SIMID", "VENDOREVENTID"]).sum()
    if dup:
        raise ValueError(f"{dup:,} duplicate (SIMID, VENDOREVENTID) rows -- the vendor "
                         "map is not 1:1 at this HAZARDZONEID/VENDORVERID; losses "
                         "would be multiply counted")
    print(f"mapping: {len(df):,} rows | {df['SIMID'].nunique():,} sims | "
          f"{df['VENDOREVENTID'].nunique():,} vendor events | "
          f"{df['EVENTID'].nunique():,} QBE events | "
          f"{len(df) / df['VENDOREVENTID'].nunique():.1f} sims/event")
    return df


def load_account_elt(account_id=None, anlsid=ANLSID_ACCOUNT, persp=PERSPCODE):
    """RMS main. One account (or all, if account_id is None)."""
    id_clause = f"AND ra.ID = {account_id}" if account_id is not None else ""
    df = pd.read_sql(SQL_ACCOUNT_ELT.format(anlsid=anlsid, persp=persp,
                                            id_clause=id_clause), engine_rdm)
    if account_id is not None and df["VENDOREVENTID"].duplicated().any():
        raise ValueError(f"ELT not unique on event for id={account_id} -- check "
                         "PARTITIONID / PERSPCODE filters")
    return df


def load_group_elt(table=None, where=None, persp=PERSPCODE):
    """
    QBE main. The GROUP portfolio ELT -- what the marginal is measured against.

    Column names on QBE main will not match the RMS ones. Rename to the canonical
    set (PERSPVALUE, STDDEVC, STDDEVI, EXPVALUE, RATE) and set GROUP_EVENT_KEY so
    the rest of the pipeline is source-agnostic.
    """
    table = table or GROUP_TABLE
    if table is None:
        raise ValueError("set GROUP_TABLE to the group ELT table/view on QBE main")
    where = GROUP_FILTER if where is None else where

    df = pd.read_sql(f"SELECT * FROM {table} WHERE 1=1 {where}", engine_qbe)
    print(f"group ELT: {len(df):,} rows | columns: {list(df.columns)}")
    return df


def inspect_group_key(group_elt: pd.DataFrame, mapping: pd.DataFrame,
                      candidates=None) -> pd.DataFrame:
    """
    Decide the group ELT's event key FROM THE DATA, not by assumption.

    RMS event ids live in VENDOREVENTID space; QBE-native ids live in EVENTID
    space. A wrong guess does not error -- it silently produces an empty or
    partial join. Set GROUP_EVENT_KEY to the column whose match rate is ~100%.
    """
    vendor_space = set(mapping["VENDOREVENTID"].unique())
    qbe_space    = set(mapping["EVENTID"].unique())

    if candidates is None:
        candidates = [c for c in group_elt.columns
                      if "EVENT" in c.upper() or c.upper().endswith("ID")]

    rows = []
    for col in candidates:
        vals = group_elt[col].dropna()
        if not pd.api.types.is_numeric_dtype(vals) or vals.empty:
            continue
        rows.append({
            "column": col,
            "n_distinct": vals.nunique(),
            "match_VENDOREVENTID": vals.isin(vendor_space).mean(),
            "match_EVENTID":       vals.isin(qbe_space).mean(),
            "min": int(vals.min()), "max": int(vals.max()),
        })
    out = pd.DataFrame(rows)
    print("Set GROUP_EVENT_KEY from the column matching ~100% in exactly one space.")
    print("A column matching partially in both is ambiguous -- do not use it.")
    return out

## 3 · Beta fit`sigma = (stdc + stdi) / expvalue` — the standard deviations are **added**, notcombined in quadrature. That is deliberate and follows from the data model: YEQTcarries one `LossQuantile` per (SimID, EventID), shared across every account, sosecondary uncertainty is comonotonic by construction and the independent componentdoes no diversifying work anywhere in the chain. Worth stating explicitly in themethodology doc — it makes portfolio volatility conservative.Where `sigma² ≥ mu(1-mu)` the moment match is infeasible; `clean_params` floorsalpha, which preserves the mean and caps variance at its theoretical maximum. Thatis the right degradation, but it is a real change in shape, so the fraction of AALaffected is reported rather than swallowed.

In [3]:
@dataclass
class BetaFitReport:
    n_rows: int = 0
    n_floored: int = 0
    n_mu_clipped: int = 0
    n_zero_exposure: int = 0
    aal_floored_share: float = 0.0
    def __str__(self):
        return (f"beta fit: {self.n_rows:,} rows | floored {self.n_floored:,} "
                f"({self.aal_floored_share:.2%} of AAL) | mu clipped {self.n_mu_clipped:,} "
                f"| zero exposure {self.n_zero_exposure:,}")


def beta_params(mean, expvalue, stdc, stdi):
    rep = BetaFitReport(n_rows=len(mean))
    mean     = np.asarray(mean, float)
    expvalue = np.asarray(expvalue, float)
    stdc     = np.asarray(stdc, float)
    stdi     = np.asarray(stdi, float)

    bad_exp = expvalue <= 0
    rep.n_zero_exposure = int(bad_exp.sum())
    safe_exp = np.where(bad_exp, 1.0, expvalue)

    mu = mean / safe_exp
    rep.n_mu_clipped = int((mu >= 1.0).sum())
    mu = np.clip(mu, PARAM_FLOOR, MU_CAP)

    sigma = np.maximum((stdc + stdi) / safe_exp, PARAM_FLOOR)

    common = (mu * (1.0 - mu)) / sigma**2 - 1.0
    alpha, beta = mu * common, (1.0 - mu) * common

    infeasible = (alpha <= 0) | (beta <= 0) | ~np.isfinite(alpha) | ~np.isfinite(beta)
    rep.n_floored = int(infeasible.sum())
    if rep.n_floored and mean.sum() > 0:
        rep.aal_floored_share = float(mean[infeasible].sum() / mean.sum())

    a = np.maximum(PARAM_FLOOR, alpha)
    b = np.maximum(PARAM_FLOOR, a * ((1.0 - mu) / mu))
    hit_b = b <= PARAM_FLOOR
    a = np.where(hit_b, np.maximum(PARAM_FLOOR, (mu * b) / (1.0 - mu)), a)

    a = np.where(np.isfinite(a) & (a > 0), a, PARAM_FLOOR)
    b = np.where(np.isfinite(b) & (b > 0), b, PARAM_FLOOR)
    a = np.where(bad_exp, PARAM_FLOOR, a)
    b = np.where(bad_exp, PARAM_FLOOR, b)

    assert np.all(a > 0) and np.all(b > 0), "invalid beta params after cleaning"
    return a, b, rep

## 4 · ELT → YLT

One function for **both** ELTs, despite their coming from different servers. It
takes the event key explicitly (`key=`), so the account ELT joins through
`VENDOREVENTID` and the group ELT joins through whichever space
`inspect_group_key` identified. Output is always keyed on the QBE
`(SIMID, EVENTID)` space, which is what makes the step-5 subtraction valid.

Losses are a deterministic function of the shared `LossQuantile`, so common random
numbers are exact and free — no reseeding, no CRN removal, no bootstrap SEs on the
difference.

`RATE` is not used here. Frequency comes entirely from the YEQT occurrence
structure; applying the rate again would double-count it.

**One trap when keying on `EVENTID`.** The mapping has one row per
(SIMID, VENDOREVENTID). If several vendor events map to the same QBE `EVENTID`,
joining an EVENTID-keyed ELT to it fans out. `simulate_elt` collapses the mapping
to unique (SIMID, EVENTID) first and reports how many rows that removed.

In [4]:
def simulate_elt(elt: pd.DataFrame, mapping: pd.DataFrame,
                 key: str = "VENDOREVENTID",
                 sims: np.ndarray | None = None,
                 loss_name: str = "Loss") -> tuple[pd.DataFrame, BetaFitReport]:
    '''
    elt     : <key>, PERSPVALUE, STDDEVC, STDDEVI, EXPVALUE  (unique on <key>)
    mapping : SIMID, VENDOREVENTID, EVENTID, LossQuantile
    key     : "VENDOREVENTID" (RMS account ELT) or "EVENTID" (QBE-native group ELT)
    sims    : optional restriction to a set of SimIDs (the band)

    Returns event-level losses on the QBE (SIMID, EVENTID) space, whichever key
    was used -- so account and group losses are directly subtractable.
    '''
    if key not in ("VENDOREVENTID", "EVENTID"):
        raise ValueError(f"key must be VENDOREVENTID or EVENTID, got {key!r}")
    missing = [c for c in [key] + ELT_COLS if c not in elt.columns]
    if missing:
        raise ValueError(f"ELT missing columns {missing} -- rename to the canonical set")
    if elt[key].duplicated().any():
        raise ValueError(f"ELT not unique on {key} -- aggregate or tighten the filter "
                         "before simulating")

    q = mapping if sims is None else mapping[mapping["SIMID"].isin(sims)]

    if key == "EVENTID":
        # several vendor events can share one QBE EVENTID; collapse so the join
        # cannot fan out. LossQuantile is a property of (SIMID, EVENTID).
        before = len(q)
        q = q.drop_duplicates(subset=["SIMID", "EVENTID"])
        if before != len(q):
            print(f"  note: collapsed {before - len(q):,} duplicate (SIMID, EVENTID) "
                  f"mapping rows for an EVENTID-keyed join")

    j = q.merge(elt, on=key, how="inner", validate="m:1")
    if j.empty:
        return pd.DataFrame(columns=["SIMID", "EVENTID", loss_name]), BetaFitReport()

    a, b, rep = beta_params(j["PERSPVALUE"].to_numpy(), j["EXPVALUE"].to_numpy(),
                            j["STDDEVC"].to_numpy(),   j["STDDEVI"].to_numpy())
    j[loss_name] = j["EXPVALUE"].to_numpy() * stats.beta.ppf(
        j["LossQuantile"].to_numpy(), a, b)

    if sims is None:
        # only meaningful on a full-mapping call; when `sims` restricts to the band,
        # most events legitimately do not occur in those particular years
        matched = elt[key].isin(set(mapping[key])).mean()
        if matched < 0.99:
            print(f"  note: only {matched:.1%} of ELT events found in the mapping "
                  f"(key={key}) -- check GROUP_EVENT_KEY / VENDORVERID / HAZARDZONEID")

    return j[["SIMID", "EVENTID", loss_name]], rep


def reconcile_aal(elt: pd.DataFrame, ylt: pd.DataFrame, n_sims: int,
                  loss_name: str = "Loss", rate_col: str = "RATE") -> dict:
    '''
    The ONLY use of RATE. Analytic AAL = sum(rate * perspvalue) over the full ELT;
    simulated AAL = total simulated loss / n_sims over the FULL (unwindowed) YLT.
    Divergence implies a mapping error or the wrong VENDORVERID -- the single
    strongest control here, since frequency is otherwise trusted.

    If the group ELT carries no rate column, this control is unavailable on the
    group side; say so rather than silently skipping it.
    '''
    if rate_col not in elt.columns:
        return {"aal_analytic": None, "aal_simulated": float(ylt[loss_name].sum()) / n_sims,
                "rel_diff": None,
                "note": f"no {rate_col} column -- analytic AAL control unavailable"}
    analytic  = float((elt[rate_col] * elt["PERSPVALUE"]).sum())
    simulated = float(ylt[loss_name].sum()) / n_sims
    return {"aal_analytic": analytic, "aal_simulated": simulated,
            "rel_diff": abs(simulated - analytic) / analytic if analytic else np.nan}

## 5 · Steps 1-3 · Portfolio YLT and the band window

In [5]:
@dataclass
class Window:
    lo: int                    # first rank in the band
    hi: int                    # last  rank in the band
    band_pos: int              # target rank's position within the band
    target_rank: int
    n_sims: int             # from the sim table, including loss-free years
    n_loss_years: int       # years that appear in the YLT
    original_rp_loss: float
    year_oep: pd.Series        # every simulated year
    band_years: np.ndarray     # SimIDs at ranks lo..hi
    head_years: np.ndarray     # SimIDs at ranks 1..lo-1  (diagnostic only)
    events: pd.DataFrame       # SIMID, EVENTID, Loss for band years -- ALL events
    edge_lo_loss: float        # original OEP at rank lo-1
    edge_hi_loss: float        # original OEP at rank hi+1

    def __str__(self):
        return (f"band [{self.lo:,}, {self.hi:,}] | {self.n_sims:,} sim years "
                f"({self.n_loss_years:,} with loss) | "
                f"target rank {self.target_rank:,} (pos {self.band_pos:,} in band) | "
                f"original {RETURN_PERIOD}RP OEP {self.original_rp_loss:,.0f} | "
                f"edges {self.edge_lo_loss:,.0f} .. {self.edge_hi_loss:,.0f} | "
                f"{len(self.events):,} (year,event) rows")


def build_window(portfolio_ylt: pd.DataFrame, n_sims: int,
                 lo: int = BAND_LO, hi: int = BAND_HI,
                 rp: int = RETURN_PERIOD) -> Window:
    '''
    Window is the BAND of ranks [lo, hi], per the agreed methodology.
    Band years carry ALL their events -- required because removing an account can
    demote a year's largest event below another event in the same year.
    '''
    year_oep = portfolio_ylt.groupby("SIMID")["Loss"].max()
    rank, _ = band_geometry(n_sims, rp, lo, hi)

    # loss-free years are absent from the YLT but are real simulated years. They
    # sit at the bottom of the ranking, so they cannot affect ranks in the band --
    # but they MUST be counted when deriving the rank itself.
    n_loss_years = len(year_oep)
    if n_loss_years > n_sims:
        raise ValueError(f"YLT has {n_loss_years:,} years but the sim table reports "
                         f"{n_sims:,} -- wrong simulation set")
    if hi >= n_loss_years:
        raise ValueError(f"band top {hi:,} exceeds the {n_loss_years:,} years that "
                         "carry any loss")

    ordered = year_oep.sort_values(ascending=False)
    band_years = ordered.index[lo - 1:hi].to_numpy()
    head_years = ordered.index[:lo - 1].to_numpy()

    events = portfolio_ylt[portfolio_ylt["SIMID"].isin(band_years)][
        ["SIMID", "EVENTID", "Loss"]].copy()

    return Window(
        lo=lo, hi=hi, band_pos=rank - (lo - 1), target_rank=rank, n_sims=n_sims,
        n_loss_years=n_loss_years,
        original_rp_loss=float(ordered.iloc[rank - 1]),
        year_oep=year_oep, band_years=band_years, head_years=head_years,
        events=events,
        edge_lo_loss=float(ordered.iloc[lo - 2]) if lo >= 2 else float("inf"),
        edge_hi_loss=float(ordered.iloc[hi]))

## 6 · Steps 4-8 · Subtract, rerank, read offAfter subtraction the band is re-sorted and the RP loss is read at position`BAND_POS` within it — i.e. the 1,501st of the 3,000 band years, which is overallrank 2,500 on the assumption that ranks 1..999 stay above it.Three estimators are reported. `point` is the methodology as written; the other twoare the smoothing variants from the note. Smoothing is applied **identically** tothe original and post-removal curves, otherwise the smoothing itself wouldcontaminate the difference.| estimator | definition ||---|---|| `point` | value at `BAND_POS` || `band_mean` | mean over all band years || `proximity` | triangular-weighted mean, peak at `BAND_POS` |**A structural point.** Portfolio and account are fitted as *separate* betas andevaluated at the *same* quantile, so `account_loss(q) > portfolio_loss(q)` canoccur row by row — the same non-linearity as`Simulated(P) - Simulated(A) != Simulated(P - A)`. Negatives are clipped and counted.**Assumption diagnostics** (reported, never altering the marginal):`head_breaches` counts years from ranks 1..999 that fell below the band answer, and`tail_breach` flags whether the band answer dropped below the original OEP at rank`hi+1`. Either being non-zero means a year outside the band belonged inside it.

In [6]:
def _proximity_weights(n: int, pos: int) -> np.ndarray:
    '''Triangular weights peaking at 1-based position `pos`, zero at the edges.'''
    idx = np.arange(1, n + 1)
    w = 1.0 - np.abs(idx - pos) / max(pos - 1, n - pos, 1)
    return np.maximum(w, 0.0) / np.maximum(w, 0.0).sum()


@dataclass
class MarginalResult:
    account_id: object
    original_rp_loss: float
    post_removal_rp_loss: float
    marginal: float
    marginal_band_mean: float
    marginal_proximity: float
    lo: int
    hi: int
    n_years_touched: int = 0
    n_demotions: int = 0
    n_negative_net: int = 0
    clipped_amount: float = 0.0
    head_breaches: int = 0
    tail_breach: bool = False
    fit_report: BetaFitReport | None = None
    notes: list = field(default_factory=list)

    def __str__(self):
        warn = ""
        if self.head_breaches or self.tail_breach:
            warn = (f" | ASSUMPTION BREACH: head {self.head_breaches:,}"
                    f"{', tail' if self.tail_breach else ''}")
        return (f"{self.account_id}: marginal {self.marginal:,.0f} "
                f"({self.original_rp_loss:,.0f} -> {self.post_removal_rp_loss:,.0f}) "
                f"| band-mean {self.marginal_band_mean:,.0f} "
                f"| proximity {self.marginal_proximity:,.0f} "
                f"| touched {self.n_years_touched:,} | demotions {self.n_demotions:,}"
                f"{warn}")


def marginal_impact(window: Window, account_losses: pd.DataFrame,
                    account_id=None, head_losses: pd.DataFrame | None = None
                    ) -> MarginalResult:
    ev = window.events.rename(columns={"Loss": "PortfolioLoss"})

    if account_losses.empty:
        m = ev.assign(AccountLoss=0.0)
    else:
        m = ev.merge(account_losses.rename(columns={"Loss": "AccountLoss"}),
                     on=["SIMID", "EVENTID"], how="left", validate="1:1")
        m["AccountLoss"] = m["AccountLoss"].fillna(0.0)

    notes = []
    over = m["AccountLoss"] > m["PortfolioLoss"] + TOL
    n_neg = int(over.sum())
    clipped = float((m.loc[over, "AccountLoss"] - m.loc[over, "PortfolioLoss"]).sum())
    if n_neg:
        notes.append(f"{n_neg:,} rows where account loss exceeded portfolio loss "
                     f"(total {clipped:,.0f}) -- clipped to zero net")
        m.loc[over, "AccountLoss"] = m.loc[over, "PortfolioLoss"]

    m["NetLoss"] = m["PortfolioLoss"] - m["AccountLoss"]

    # per-year max over the FULL event set -- demotion-safe
    agg = m.groupby("SIMID").agg(orig_max=("PortfolioLoss", "max"),
                                 net_max=("NetLoss", "max"),
                                 orig_arg=("PortfolioLoss", "idxmax"),
                                 net_arg=("NetLoss", "idxmax"))
    assert (agg["net_max"] <= agg["orig_max"] + TOL).all(), "removal raised a year loss"

    net_sorted  = np.sort(agg["net_max"].to_numpy())[::-1]
    orig_sorted = np.sort(agg["orig_max"].to_numpy())[::-1]
    n = len(net_sorted)
    pos = min(window.band_pos, n)

    post_point = float(net_sorted[pos - 1])
    orig_point = float(orig_sorted[pos - 1])

    w = _proximity_weights(n, pos)
    post_prox, orig_prox = float(net_sorted @ w), float(orig_sorted @ w)
    post_mean, orig_mean = float(net_sorted.mean()), float(orig_sorted.mean())

    # --- assumption diagnostics (do NOT change the reported marginal) -------
    head_breaches = 0
    if head_losses is not None and len(window.head_years):
        h = head_losses.groupby("SIMID")["NetLoss"].max()
        head_breaches = int((h < post_point).sum())
        if head_breaches:
            notes.append(f"{head_breaches:,} of {len(window.head_years):,} head years "
                         f"(ranks 1..{window.lo - 1:,}) fell below the band answer -- "
                         "they belong inside the window")

    tail_breach = post_point < window.edge_hi_loss - TOL
    if tail_breach:
        notes.append(f"band answer {post_point:,.0f} < original OEP at rank "
                     f"{window.hi + 1:,} ({window.edge_hi_loss:,.0f}) -- years below "
                     "the band could enter the top " f"{window.target_rank:,}")

    return MarginalResult(
        account_id=account_id,
        original_rp_loss=orig_point,
        post_removal_rp_loss=post_point,
        marginal=orig_point - post_point,
        marginal_band_mean=orig_mean - post_mean,
        marginal_proximity=orig_prox - post_prox,
        lo=window.lo, hi=window.hi,
        n_years_touched=int((agg["orig_max"] - agg["net_max"] > TOL).sum()),
        n_demotions=int((agg["orig_arg"] != agg["net_arg"]).sum()),
        n_negative_net=n_neg, clipped_amount=clipped,
        head_breaches=head_breaches, tail_breach=tail_breach, notes=notes)

## 7 · Driver`verify_head=True` additionally simulates the account across ranks 1..999 purelyto test the methodology's assumption. It costs ~1,000 extra years per account anddoes **not** affect the reported marginal — it only populates `head_breaches`, whichis the evidence needed for the later analysis phase.

In [7]:
def run_account(account_elt: pd.DataFrame, mapping: pd.DataFrame, window: Window,
                account_id=None, group_ylt: pd.DataFrame | None = None,
                key: str = "VENDOREVENTID",
                verify_head: bool = True) -> MarginalResult:
    '''
    account_elt : RMS main, keyed on VENDOREVENTID
    mapping     : QBE sandbox, already restricted to band (+ head) years
    window      : built from the GROUP portfolio YLT
    group_ylt   : needed only for the head diagnostic
    '''
    losses, fit = simulate_elt(account_elt, mapping, key=key, sims=window.band_years)

    head_net = None
    if verify_head and group_ylt is not None and len(window.head_years):
        head_pf = group_ylt[group_ylt["SIMID"].isin(window.head_years)]
        head_ac, _ = simulate_elt(account_elt, mapping, key=key,
                                  sims=window.head_years)
        h = head_pf.rename(columns={"Loss": "PortfolioLoss"}).merge(
            head_ac.rename(columns={"Loss": "AccountLoss"}),
            on=["SIMID", "EVENTID"], how="left")
        h["AccountLoss"] = h["AccountLoss"].fillna(0.0).clip(upper=h["PortfolioLoss"])
        h["NetLoss"] = h["PortfolioLoss"] - h["AccountLoss"]
        head_net = h

    res = marginal_impact(window, losses, account_id=account_id, head_losses=head_net)
    res.fit_report = fit
    return res

## 8 · Production run

Order matters. The group YLT spans **all** years and cannot be windowed — the
200 RP is defined on the full distribution. The band is built once and reused for
every account.

Step 1b is not optional: run `inspect_group_key` and set `GROUP_EVENT_KEY` from
what it reports. A wrong key produces an empty or partial join, not an error.

**Expect the clipping diagnostic to matter more now.** The account sits inside an
RMS analysis; the group portfolio is assembled on QBE main. There is no guarantee
the account is a strict subset of the group, and the two ELTs are fitted as
separate betas evaluated at the same quantile. Watch `clipped_rows` and
`clipped_amount` on the first run — a large clipped amount means the account is
not really inside that group, or the perspectives differ.

In [8]:
# --- 0. load once ----------------------------------------------------------
# N_SIMS, SIM_LO, SIM_HI = load_n_sims()          # QBE sandbox; derived, not assumed
# TARGET_RANK, BAND_POS  = band_geometry(N_SIMS)
# print(f"target rank {TARGET_RANK:,}, position {BAND_POS:,} in the band")
#
# mapping_df = load_mapping()                     # QBE sandbox

# --- 1a. GROUP portfolio ELT (QBE main) ------------------------------------
# group_elt = load_group_elt()
#
# --- 1b. decide its event key FROM THE DATA --------------------------------
# display(inspect_group_key(group_elt, mapping_df))
# GROUP_EVENT_KEY = "EVENTID"        # <- set from the table above, then rerun
#
# rename QBE main's columns to the canonical set, then:
# group_elt = group_elt.rename(columns={...})
# group_elt = group_elt.groupby(GROUP_EVENT_KEY, as_index=False)[ELT_COLS].sum()
#   ^ only if the group ELT is not already unique on its event key

# --- 1c. group ELT -> group YLT (all years) --------------------------------
# group_ylt, g_fit = simulate_elt(group_elt, mapping_df, key=GROUP_EVENT_KEY)
# print(g_fit)
# print(reconcile_aal(group_elt, group_ylt, N_SIMS))     # None if no RATE column

# --- 2-3. band window, once ------------------------------------------------
# window = build_window(group_ylt, n_sims=N_SIMS)        # ranks BAND_LO..BAND_HI
# print(window)
#
# restrict the mapping ONCE -- band years plus head years (head is diagnostic only)
# keep = set(window.band_years) | set(window.head_years)
# mapping_win = mapping_df[mapping_df["SIMID"].isin(keep)]
# print(f"mapping restricted: {len(mapping_win):,} rows "
#       f"({len(mapping_win)/len(mapping_df):.1%} of full)")

# --- 4-8. loop accounts (RMS main) -----------------------------------------
# account_ids = sorted(load_account_elt()["ID"].unique())     # or an explicit list
# results = []
# for acct in account_ids:
#     elt = load_account_elt(account_id=acct)
#     res = run_account(elt, mapping_win, window, acct, group_ylt=group_ylt)
#     results.append(res)
#     print(res)

# summary = pd.DataFrame([{
#     "account": r.account_id, "marginal": r.marginal,
#     "marginal_band_mean": r.marginal_band_mean,
#     "marginal_proximity": r.marginal_proximity,
#     "years_touched": r.n_years_touched, "demotions": r.n_demotions,
#     "clipped_rows": r.n_negative_net, "clipped_amount": r.clipped_amount,
#     "head_breaches": r.head_breaches, "tail_breach": r.tail_breach,
# } for r in results]).sort_values("marginal", ascending=False)
# summary
print("fill in the three connections and GROUP_EVENT_KEY above")

fill in the three connections and GROUP_EVENT_KEY above


## 9 · Self-test — synthetic three-server universe

Runs without any server. The group ELT is built in **QBE `EVENTID` space** and the
account ELTs in **RMS `VENDOREVENTID` space**, so the dual-key path is exercised
exactly as in production. Three things are measured:

1. `inspect_group_key` picks the right key from the data;
2. the engine does what the methodology says (band rerank, demotion-safe max);
3. **how far the band answer sits from the true answer**, by brute force over every
   simulated year. That gap is the methodology's approximation error, recorded as
   evidence for the analysis phase — not a code defect.

In [9]:
rng = np.random.default_rng(20260925)
_E, _RP = 400, 200
_SIM_TABLE = np.arange(1, 50_001)
_N = len(_SIM_TABLE)                         # derived, not assumed
_RANK = int(round(_N / _RP))                 # 250
_LO, _HI = 100, 400                          # band scaled to _N, same shape as 1000-4000

_ev = pd.DataFrame({"VENDOREVENTID": np.arange(1, _E + 1),
                    "EVENTID": np.arange(90_001, 90_001 + _E)})

_occ = np.array([(s, e) for s in _SIM_TABLE
                 for e in rng.choice(_E, size=min(rng.poisson(1.2), 4), replace=False)])
_map = pd.DataFrame({"SIMID": _occ[:, 0],
                     "VENDOREVENTID": _ev["VENDOREVENTID"].to_numpy()[_occ[:, 1]],
                     "EVENTID":       _ev["EVENTID"].to_numpy()[_occ[:, 1]],
                     "LossQuantile":  rng.uniform(1e-6, 1 - 1e-6, len(_occ))})

# --- account ELTs: RMS main, VENDOREVENTID space ---------------------------
def _mk(n, scale, seed):
    r = np.random.default_rng(seed)
    idx  = r.choice(_E, size=n, replace=False)
    exp  = r.uniform(1e5, 5e6, n) * scale
    mean = exp * r.uniform(0.001, 0.25, n)
    return pd.DataFrame({"VENDOREVENTID": _ev["VENDOREVENTID"].to_numpy()[idx],
                         "PERSPVALUE": mean, "EXPVALUE": exp,
                         "STDDEVC": mean * r.uniform(0.2, 1.5, n),
                         "STDDEVI": mean * r.uniform(0.2, 2.0, n),
                         "RATE": r.uniform(1e-6, 5e-4, n)})

_accts = {"A_big": _mk(300, 6.0, 1), "B_mid": _mk(200, 1.0, 2),
          "C_small": _mk(60, 0.2, 3), "D_tiny": _mk(10, 0.02, 4)}

# --- GROUP ELT: QBE main, EVENTID space, and WIDER than the accounts -------
_grp = (pd.concat(_accts.values()).groupby("VENDOREVENTID", as_index=False)
          .agg(PERSPVALUE=("PERSPVALUE", "sum"), EXPVALUE=("EXPVALUE", "sum"),
               STDDEVC=("STDDEVC", "sum"), STDDEVI=("STDDEVI", "sum")))
_grp = _grp.merge(_ev, on="VENDOREVENTID").drop(columns="VENDOREVENTID")
_grp[ELT_COLS] *= 1.35        # the group holds business beyond these four accounts

print("--- 1. which key does the group ELT use? ---")
display(inspect_group_key(_grp, _map))
_KEY = "EVENTID"              # read off the table above

print("\n--- 2. group YLT and band ---")
_grp_ylt, _g_fit = simulate_elt(_grp, _map, key=_KEY)
print(_g_fit)
print(f"sim table: {_N:,} years | YLT covers {_grp_ylt['SIMID'].nunique():,} "
      f"({_grp_ylt['SIMID'].nunique()/_N:.1%}) -- the rest are loss-free")

_win = build_window(_grp_ylt, n_sims=_N, lo=_LO, hi=_HI)
print(_win, "\n")

print("--- 3. account marginals against the group ---")
_res = {}
for _a, _e in _accts.items():
    _r = run_account(_e, _map, _win, _a, group_ylt=_grp_ylt)
    _res[_a] = _r
    print(_r)
    for _n in _r.notes:
        print("    NOTE:", _n)

--- 1. which key does the group ELT use? ---
Set GROUP_EVENT_KEY from the column matching ~100% in exactly one space.
A column matching partially in both is ambiguous -- do not use it.


,column,n_distinct,match_VENDOREVENTID,match_EVENTID,min,max
0,EVENTID,351,0.00,1.00,90001,90400



--- 2. group YLT and band ---
beta fit: 52,009 rows | floored 12,264 (33.66% of AAL) | mu clipped 0 | zero exposure 0
sim table: 50,000 years | YLT covers 32,485 (65.0%) -- the rest are loss-free
band [100, 400] | 50,000 sim years (32,485 with loss) | target rank 250 (pos 151 in band) | original 200RP OEP 38,396,675 | edges 42,734,914 .. 36,679,153 | 599 (year,event) rows 

--- 3. account marginals against the group ---
A_big: marginal 28,278,479 (38,396,675 -> 10,118,196) | band-mean 27,511,472 | proximity 27,804,157 | touched 301 | demotions 1 | ASSUMPTION BREACH: head 0, tail
    NOTE: 17 rows where account loss exceeded portfolio loss (total 1,706,327) -- clipped to zero net
    NOTE: band answer 10,118,196 < original OEP at rank 401 (36,679,153) -- years below the band could enter the top 250
B_mid: marginal 913,785 (38,396,675 -> 37,482,890) | band-mean 720,934 | proximity 598,725 | touched 109 | demotions 0
    NOTE: 31 rows where account loss exceeded portfolio loss (total 1,0

In [10]:
print("--- engine invariants ---")

assert all(r.marginal >= -TOL for r in _res.values())
print("1. all marginals non-negative                      PASS")

assert all(r.marginal <= r.original_rp_loss + TOL for r in _res.values())
print("2. marginals bounded by original RP loss           PASS")

assert all(r.lo == _LO and r.hi == _HI for r in _res.values())
print("3. band respected                                  PASS")

# the group ELT is EVENTID-keyed, the accounts VENDOREVENTID-keyed; both land in
# the same (SIMID, EVENTID) space or the subtraction below would be meaningless
_a_ylt, _ = simulate_elt(_accts["A_big"], _map, key="VENDOREVENTID")
assert set(_a_ylt["EVENTID"]).issubset(set(_grp_ylt["EVENTID"]))
print("4. dual-key outputs share one event space          PASS")

print("\n--- methodology error vs brute force over ALL years ---")
rows = []
for _a, _e in _accts.items():
    _full, _ = simulate_elt(_e, _map, key="VENDOREVENTID")
    _m = _grp_ylt.merge(_full.rename(columns={"Loss": "A"}),
                        on=["SIMID", "EVENTID"], how="left")
    _m["A"] = _m["A"].fillna(0.0).clip(upper=_m["Loss"])
    _m["Net"] = _m["Loss"] - _m["A"]
    _true_post = float(np.sort(_m.groupby("SIMID")["Net"].max().to_numpy())[::-1][_RANK - 1])
    _true_marg = _win.original_rp_loss - _true_post
    _band_marg = _res[_a].marginal
    rows.append({"account": _a, "band_marginal": _band_marg,
                 "true_marginal": _true_marg,
                 "abs_error": _band_marg - _true_marg,
                 "rel_error": (_band_marg - _true_marg) / _true_marg if _true_marg else np.nan,
                 "head_breaches": _res[_a].head_breaches,
                 "tail_breach": _res[_a].tail_breach})
err = pd.DataFrame(rows)
display(err)
print("\nband_marginal is the methodology as specified; true_marginal is the exact")
print("answer. Non-zero error comes from years outside [lo, hi] that should have")
print("been inside -- head_breaches and tail_breach identify them.")

--- engine invariants ---
1. all marginals non-negative                      PASS
2. marginals bounded by original RP loss           PASS
3. band respected                                  PASS


4. dual-key outputs share one event space          PASS

--- methodology error vs brute force over ALL years ---


,account,band_marginal,true_marginal,abs_error,rel_error,head_breaches,tail_breach
0,A_big,"28,278,478.51","25,704,207.10","2,574,271.41",0.10,0,True
1,B_mid,"913,784.59","913,784.59",0.00,0.00,0,False
2,C_small,0.00,0.00,0.00,NaN,0,False
3,D_tiny,0.00,0.00,0.00,NaN,0,False



band_marginal is the methodology as specified; true_marginal is the exact
answer. Non-zero error comes from years outside [lo, hi] that should have
been inside -- head_breaches and tail_breach identify them.


## 10 · Controls to run before reporting any number

| control | where | fails if |
|---|---|---|
| Group event key match ~100% in one space | `inspect_group_key` | wrong `GROUP_EVENT_KEY` → empty or partial join |
| Map 1:1 on `(SIMID, VENDOREVENTID)` | `load_mapping` | raises; losses would be multiply counted |
| Account ELT unique on event | `load_account_elt` | raises |
| Group ELT unique on its key | `simulate_elt` | raises |
| ELT events found in mapping | `simulate_elt` | prints a note below 99% |
| AAL analytic vs simulated | `reconcile_aal` | wrong `VENDORVERID` / mapping gap |
| `net_max ≤ orig_max` | `marginal_impact` | asserts |
| Clipped-row count ≈ 0 | `marginal_impact` | account not inside this group, or perspective mismatch |
| `head_breaches` / `tail_breach` = 0 | `marginal_impact` | band too narrow; the answer is approximate |
| Both ELTs at the same perspective | manual | marginals won't reconcile to the group |
| Sim count from SIMID **range** | `load_n_sims` | loss-free years dropped → wrong target rank |

**Open questions for Tim**

1. **Perspective on the group ELT.** `RL` is set on the RMS side. Confirm the QBE
   main group ELT is the same perspective, or the subtraction mixes bases.
2. **Is each account actually inside the group portfolio?** The clipping counter
   answers this empirically on the first run.
3. **The band.** Ranks 1–999 are excluded, but removal only lowers losses, so the
   post-removal rank-2,500 year can rise from anywhere above it. `head_breaches`
   counts how often that happened. If it's non-zero, the band understates the
   marginal for exactly the tail-concentrated accounts that matter most.
4. **Expect exact zeros.** Any account that never touches a band year's maximum
   event scores 0. Report the zero rate as a headline diagnostic.
5. **Non-additivity.** Last-in marginals on a quantile don't sum to the group
   200 RP. Pair with co-TVaR/Euler if anyone wants an additive view.